In [15]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch import tensor

import numpy as np
import pandas as pd

# First attempt outline

Dataset:
- create custom pytorch dataset object that is then compatible with data_loader

Want to create a neural network that predicts a single genes expression across samples from all TFs
- will not subset to network yet to get hang of pytorch
- will not use MML yet to get hang of pytorch

X: Expression of all TFs in all samples
Y: Expression of single gene in all samples

Activation function: linear activation function



### Creating dataset object

In [16]:
#torch tutorial code to use accelerator when available
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [17]:
#DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'
#TF_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)
#TF_expressions.shape

In [18]:
from torch.utils.data import Dataset


class CustomTFGE(Dataset):
    def __init__(self, device, transform=None, target_transform=None):
        #load the two tsv files
        self.DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'
        self.TF_expressions = pd.read_csv((f"{self.DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)
        self.gene_expressions = pd.read_csv((f"{self.DATA_ROOT}/Full data files/Geneexpression (full).tsv"), sep='\t', header=0)

        #no transforms needed so set to none
        self.transform = transform
        self.target_transform = target_transform

        #convert to torch tensors
        self.TF_expressions = torch.tensor(np.asarray(self.TF_expressions).T, dtype = torch.float32, device = device)
        self.gene_expressions = torch.tensor(np.asarray(self.gene_expressions), dtype = torch.float32, device = device)

    def __len__(self):
        #length of the dataset is the number of samples (not TFs in the dataset) - 15935
        return self.TF_expressions.shape[0]

    def __getitem__(self, idx):
        #in this case want to always retrieve the same gene (target) but a different sample containg all the TF values
        TFs_exp = self.TF_expressions[:, idx]
        Gene_exp = self.gene_expressions[:, 1]
        #returns TFs exp and Gene_exp 
        return TFs_exp, Gene_exp

In [19]:
#initialise an instance of the dataset object - pass device so tensors and model on same device
dataset = CustomTFGE(device)
dataset

In [20]:
#new way to create train and test dataset with pytorches dataset objects
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

In [21]:
#defining the neural network
class BasicNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        #self.flatten = nn.Flatten()
        self.linear_layer = nn.Sequential(
            #linear activation layer takes 1198 TFs per sample -> outputs a value for the TF for all 15935 samples
            nn.Linear(1198, 15935)
        )

    def forward(self, x):
        #forward pass is simply the linear layer
        expressions = self.linear_layer(x)
        return(expressions)

In [22]:
#put model on same device as the tensors
model = BasicNeuralNetwork().to(device)
print(model)

BasicNeuralNetwork(
  (linear_layer): Sequential(
    (0): Linear(in_features=1198, out_features=15935, bias=True)
  )
)


## Train test loop

In [23]:
def train_loop(dataloader, model, loss_fn, optimizer):
    losses = []
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        
        loss = loss.item()
        print(f"loss: {loss:>7f}")
        losses.append(loss)
    return(losses)


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            print(f"MSE on test dataset is {loss_fn(pred, y).item()}")
            


In [24]:
#intialise hyperparameters - batch size is number of samples so that each backprop is done with the entire dataset (gradient descent not stochastic gradient descent)
#dataset is small enough for this to be fine
#investigate hyperparam tuning later
learning_rate = 1e-3
batch_size = 15935
epochs = 50

#initialize MSE loss function - same as LEMBAS
loss_fn = nn.MSELoss()

#initialise same optimiser as LEMBAS
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [25]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [26]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    losses = train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 6.395503
MSE on test dataset is 3.7198879718780518
Epoch 2
-------------------------------
loss: 3.787127
MSE on test dataset is 1.971719741821289
Epoch 3
-------------------------------
loss: 2.079668
MSE on test dataset is 1.0676319599151611
Epoch 4
-------------------------------
loss: 1.186179
MSE on test dataset is 0.804404616355896
Epoch 5
-------------------------------
loss: 0.911954
MSE on test dataset is 0.9252359867095947
Epoch 6
-------------------------------
loss: 1.010659
MSE on test dataset is 1.194764256477356
Epoch 7
-------------------------------
loss: 1.256394
MSE on test dataset is 1.4393600225448608
Epoch 8
-------------------------------
loss: 1.482400
MSE on test dataset is 1.5641933679580688
Epoch 9
-------------------------------
loss: 1.597437
MSE on test dataset is 1.5463757514953613
Epoch 10
-------------------------------
loss: 1.579186
MSE on test dataset is 1.415386438369751
Epoch 11
------------------------

# Saving model

In [27]:
torch.save(model, 'models/first_attempt_single_target_model.pth')

In [28]:
#How to load for reference
model = torch.load('models/first_attempt_single_target_model.pth', weights_only=False)